# narrative_dna Colab Quickstart

Este notebook descarga el repo desde GitHub, instala `narrative_dna`, importa los módulos principales y ejecuta un flujo JSON-first conservador desde un string o desde un `.txt`. La notación compacta se deriva siempre desde JSON validado.

## 1. Clonar el repo desde GitHub

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/jcval94/ADNarrativa.git"
REPO_DIR = Path("/content/ADNarrativa")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(Path.cwd())

## 2. Instalar el paquete

In [ ]:
%pip install -q -e ".[dev]"

## 3. Importar módulos principales

In [ ]:
import json
from pathlib import Path
import sys

REPO_DIR = Path("/content/ADNarrativa")
sys.path.insert(0, str(REPO_DIR / "src"))

from narrative_dna.evaluator import load_gold_units
from narrative_dna.loader import load_document, load_documents, load_text_document
from narrative_dna.pipeline import run_pipeline, run_pipeline_from_text

print("Imports OK")

## 4. Crear una transcripción desde un string

In [ ]:
transcript_text = """
Los aviones tienen una caja negra. Es un registro mínimo que guarda lo esencial.
Quizás podemos hacer algo parecido con nuestra memoria.
Te propongo algo simple: una vez al año, crea tu propia caja negra emocional.
¿Qué aprendiste este año? ¿Qué lograste? ¿Qué quieres dejar registrado para siempre?
""".strip()

document = load_text_document(
    transcript_text,
    document_id="colab_text_demo",
    source_path="<colab-string>",
    metadata={"source": "colab_inline_example"},
    language="es",
)

print("document_id:", document.document_id)
print("units:", len(document.units))
for unit in document.units:
    print(unit.unit_id, unit.final_notation, unit.text)

## 5. Ejecutar el pipeline JSON-first sin LLM

Este modo es conservador: genera unidades finales `N_N0{0}` y agrega heurísticas sólo como señales candidatas auditables. Para obtener etiquetas finales distintas de `N_N0{0}`, usa `use_llm=True` o una etapa de clasificación explícita.

In [ ]:
result = run_pipeline_from_text(
    transcript_text,
    document_id="colab_text_demo",
    source_path="<colab-string>",
    output_dir="outputs",
    run_id="colab_text_no_llm_demo",
    use_llm=False,
    use_adjudicator=False,
    audit_similarity_enabled=False,
)

print("run_id:", result.run_id)
print("run_dir:", result.run_dir)
print("documents:", len(result.documents))

## 6. Leer outputs JSON/JSONL

In [ ]:
run_dir = Path("outputs/colab_text_no_llm_demo")

manifest = json.loads((run_dir / "run_manifest.json").read_text(encoding="utf-8"))
print("manifest run_id:", manifest["run_id"])
print("taxonomy:", manifest["taxonomy_version"])

print("\nUnidades y candidatos heurísticos:")
for line in (run_dir / "units.jsonl").read_text(encoding="utf-8").splitlines():
    unit = json.loads(line)
    candidates = [(item["label"], round(item["confidence"], 2)) for item in unit["heuristic_candidates"]]
    print(unit["unit_id"], unit["final_notation"], candidates, unit["text"][:100])

print("\nAudit report preview:")
print((run_dir / "audit_report.md").read_text(encoding="utf-8")[:1000])

## 7. Ejecutar desde un archivo `.txt`

In [ ]:
Path("colab_example.txt").write_text(transcript_text, encoding="utf-8")
txt_result = run_pipeline(
    input_dir="colab_example.txt",
    output_dir="outputs",
    run_id="colab_txt_no_llm_demo",
    use_llm=False,
    use_adjudicator=False,
)
print(txt_result.run_dir)

## 8. Alternativa por CLI

In [ ]:
!narrative-dna run --input-dir colab_example.txt --output-dir outputs --run-id colab_cli_txt_no_llm --no-llm --no-adjudicator
!narrative-dna inspect --run-id colab_cli_txt_no_llm

## 9. Probar regresión golden

In [ ]:
!python -m pytest tests/test_golden_regression.py -q

## 10. Opcional: usar OpenAI desde Colab Secrets

En Colab, guarda `OPENAI_API_KEY` en Secrets. Después descomenta y ejecuta esta celda para clasificar con LLM y adjudicator conservador.

In [ ]:
# from google.colab import userdata
# import os
#
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
#
# llm_result = run_pipeline_from_text(
#     transcript_text,
#     document_id="colab_text_demo_llm",
#     source_path="<colab-string>",
#     output_dir="outputs",
#     run_id="colab_llm_demo",
#     use_llm=True,
#     use_adjudicator=True,
#     audit_similarity_enabled=True,
# )
# print(llm_result.run_dir)

## 11. Evaluar contra synthetic gold high-confidence

Cuando tengas `outputs/<RUN_ID>/synthetic_gold_high_confidence.jsonl`, evalúa así:

In [ ]:
# !narrative-dna evaluate --run-id colab_llm_demo --gold outputs/colab_llm_demo/synthetic_gold_high_confidence.jsonl